# Inference на 768: ймовірності U-Net з кропів для val і test

Етап 9, покращення 1, крок 3. Як `07_inference.ipynb`, але U-Net — нова (`09_unet_crops_train.ipynb`, val Dice 0.7227)
і рахується на повних 768×768, без стискання до 384. Рішення (пороги, сабміт) — локально в `08_pipeline.ipynb`.

**Чому не на всіх знімках.** Ймовірності 768 — 589 КБ на знімок, для всіх val + test ~20 ГБ у пам'яті. Тому рахуємо
лише знімки, які класифікатор пропускає з p ≥ 0.5 (у v1 оптимальний поріг був 0.9, тож запас для перебору лишається),
плюс у val — усі знімки з кораблями, щоб перевірити Dice на тих самих 4 256 знімках, що в навчанні.
Знімок, якого немає у файлі, пайплайн вважає порожнім.

| що | файл у Output |
|---|---|
| U-Net на val: p ≥ 0.5 або є кораблі | `val_unet_probs_768.npz` |
| U-Net на test: p ≥ 0.5 | `test_unet_probs_768.npz` |

Класифікатор не запускаємо — його ймовірності вже є: val — `val_probs.csv` (вихід навчання класифікатора),
test — `test_classifier_probs.csv` (вихід `07`).

Локально ноутбук проходить на `data/train_micro/` (перевірка логіки): класифікатора для цих знімків немає,
тож рахуються всі.

**На Kaggle:**
1. Settings → Accelerator → **GPU**, Internet → **On**; Add-ons → Secrets → `KAGGLE_API_TOKEN` **attached**.
2. Input: `airbus-ship-detection-src` і виходи трьох ноутбуків: **`09_unet_crops_train`** (ваги U-Net),
   **`airbus-ship-detection-outside-competition`** (`val_probs.csv`), **`airbus-inference`** (`test_classifier_probs.csv`).
   Змагання в Input **не** додавати.
3. **Save Version → Save & Run All**.

## 0. Дані на Kaggle

Як у `07`: train_v2 (для val) і test_v2.

In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
KAGGLE_DATA_DIR = Path("/kaggle/tmp/data")

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    # Встановлена в Kaggle версія kaggle може не знати нового формату токена.
    !pip install -q kaggle==2.2.4 segmentation-models-pytorch==0.5.0

In [ ]:
if ON_KAGGLE and not (KAGGLE_DATA_DIR / "test_v2").exists():
    !mkdir -p /kaggle/tmp && cd /kaggle/tmp && kaggle competitions download -c airbus-ship-detection
    !unzip -q /kaggle/tmp/airbus-ship-detection.zip "train_v2/*" "test_v2/*" train_ship_segmentations_v2.csv sample_submission_v2.csv -d {KAGGLE_DATA_DIR}

if ON_KAGGLE:
    n_train = sum(1 for _ in (KAGGLE_DATA_DIR / "train_v2").iterdir())
    n_test = sum(1 for _ in (KAGGLE_DATA_DIR / "test_v2").iterdir())
    print("знімків у train_v2:", n_train, "| у test_v2:", n_test)
    assert n_train == 192556 and n_test == 15606

## 1. Налаштування і вибір знімків

Val — та сама `make_split` (seed 42). Ймовірності класифікатора приєднуємо за `ImageId`.

In [ ]:
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
from torch.utils.data import DataLoader

if ON_KAGGLE:
    sys.path.append("/kaggle/input/datasets/yurapolishchuk/airbus-ship-detection-src")
    DATA_DIR = KAGGLE_DATA_DIR
    VAL_DIR = DATA_DIR / "train_v2"
    TEST_DIR = DATA_DIR / "test_v2"
    OUT_DIR = Path("/kaggle/working")
    UNET_WEIGHTS = next(Path("/kaggle/input").rglob("unet_crops_resnet34.pth"))
    VAL_CLASSIFIER = next(Path("/kaggle/input").rglob("val_probs.csv"))
    TEST_CLASSIFIER = next(Path("/kaggle/input").rglob("test_classifier_probs.csv"))
    DEVICE = "cuda"
    NUM_WORKERS = 4
    BATCH_SIZE = 8
else:
    ROOT = Path.cwd().parent
    sys.path.append(str(ROOT / "src"))
    DATA_DIR = ROOT / "data"
    VAL_DIR = DATA_DIR / "train_micro"
    TEST_DIR = DATA_DIR / "train_micro"
    OUT_DIR = DATA_DIR / "inference_768_local"
    UNET_WEIGHTS = DATA_DIR / "unet_crops_v1" / "unet_crops_resnet34.pth"
    DEVICE = "cpu"
    NUM_WORKERS = 0
    BATCH_SIZE = 2

from dataset import ShipSegmentationDataset, read_mask, rles_by_image
from rle import IMAGE_SHAPE
from split import make_split

SEED = 42
CLASSIFIER_MIN_P = 0.5
USE_AMP = DEVICE == "cuda"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Kaggle:", ON_KAGGLE, "| device:", DEVICE)
print("U-Net:", UNET_WEIGHTS)

In [ ]:
masks = pd.read_csv(DATA_DIR / "train_ship_segmentations_v2.csv")
rles = rles_by_image(masks)
if ON_KAGGLE:
    split = make_split(masks)
    val_df = split[split["split"] == "val"].reset_index(drop=True)
    val_df = val_df.merge(pd.read_csv(VAL_CLASSIFIER)[["ImageId", "p"]], on="ImageId", how="left")
    test_df = pd.read_csv(TEST_CLASSIFIER)
    assert val_df["p"].notna().all() and len(test_df) == 15606
else:
    available = sorted(p.name for p in VAL_DIR.iterdir())
    split = make_split(masks[masks["ImageId"].isin(available)], val_size=0.2)
    val_df = split[split["split"] == "val"].reset_index(drop=True)
    test_df = pd.DataFrame({"ImageId": available[:8]})
    # Ймовірностей класифікатора для train_micro немає — пропускаємо всі знімки.
    val_df["p"] = 1.0
    test_df["p"] = 1.0

val_ids = val_df.loc[(val_df["p"] >= CLASSIFIER_MIN_P) | (val_df["n_ships"] > 0), "ImageId"].reset_index(drop=True)
test_ids = test_df.loc[test_df["p"] >= CLASSIFIER_MIN_P, "ImageId"].reset_index(drop=True)
print(f"val: {len(val_ids)} з {len(val_df)} знімків | test: {len(test_ids)} з {len(test_df)}")
print(f"пам'ять під ймовірності: val {len(val_ids) * IMAGE_SHAPE[0] * IMAGE_SHAPE[1] / 1e9:.1f} ГБ, "
      f"test {len(test_ids) * IMAGE_SHAPE[0] * IMAGE_SHAPE[1] / 1e9:.1f} ГБ")

## 2. U-Net на val і test

Та сама функція, що в `07`, але `size=768`. Ймовірності — `uint8` (p × 255), стиснуті.

In [ ]:
unet = smp.Unet(encoder_name="resnet34", encoder_weights=None, classes=1)
unet.load_state_dict(torch.load(UNET_WEIGHTS, map_location=DEVICE))
unet = unet.to(DEVICE).eval()


@torch.no_grad()
def predict_unet(image_ids, image_dir):
    dataset = ShipSegmentationDataset(image_ids, {}, image_dir, train=False, size=IMAGE_SHAPE[0])
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    batches = []
    for images, _ in loader:
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = unet(images.to(DEVICE))
        probs = torch.sigmoid(logits.float())[:, 0]
        batches.append((probs * 255).round().to(torch.uint8).cpu().numpy())
    return np.concatenate(batches)


all_probs = {}
for name, image_ids, image_dir in [("val", val_ids, VAL_DIR), ("test", test_ids, TEST_DIR)]:
    start = time.time()
    probs = predict_unet(image_ids, image_dir)
    np.savez_compressed(OUT_DIR / f"{name}_unet_probs_768.npz", probs=probs, image_ids=np.asarray(image_ids))
    all_probs[name] = probs
    has_pixels = (probs > 0.7 * 255).any(axis=(1, 2)).mean()
    print(f"{name}: {probs.shape}, {time.time() - start:.0f} с | знімків з хоч одним пікселем p > 0.7: {has_pixels:.1%}")

for path in sorted(OUT_DIR.glob("*_unet_probs_768.npz")):
    print(f"{path.name}: {path.stat().st_size / 1e6:.0f} МБ")

### 2.1. Перевірка: Dice на val-знімках з кораблями

Має збігтися з val Dice останньої епохи навчання (0.7227 на Kaggle): ті самі знімки, ті самі ваги, поріг 0.5.
Невелика різниця (~1e-3) — нормально: AMP і округлення до `uint8`. Велика різниця означає, що
препроцесинг чи ваги не ті.

In [ ]:
def dice(predicted, mask):
    total = predicted.sum() + mask.sum()
    return 1.0 if total == 0 else 2 * (predicted & mask).sum() / total


ship_ids = set(val_df.loc[val_df["n_ships"] > 0, "ImageId"])
# p > 0.5 ⇔ округлене p × 255 ≥ 128
dices = [dice(probs >= 128, read_mask(rles[image_id], IMAGE_SHAPE[0]).astype(bool))
         for image_id, probs in zip(val_ids, all_probs["val"]) if image_id in ship_ids]
print(f"val Dice на {len(dices)} знімках з кораблями: {np.mean(dices):.4f}")

### 2.2. Перевірка очима на test

4 test-знімки з найбільшим p класифікатора: знімок і маска U-Net (p > 0.7, зелене). Розмітки для test
немає — дивимось лише, чи маска лежить на чомусь схожому на корабель.

In [ ]:
test_p = test_df.set_index("ImageId").loc[test_ids, "p"].reset_index(drop=True)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, i in zip(axes, test_p.nlargest(4).index):
    image = cv2.cvtColor(cv2.imread(str(TEST_DIR / test_ids[i])), cv2.COLOR_BGR2RGB)
    mask = all_probs["test"][i] > 0.7 * 255
    image[mask] = (0.6 * image[mask] + 0.4 * np.array([0, 255, 0])).astype(np.uint8)
    ax.imshow(image)
    ax.set_title(f"{test_ids[i]}, p = {test_p[i]:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "test_examples_768.png", dpi=80)
plt.show()